# Join swim results with environment

Load results from `swim_db.sqlite`, join with meet-level environment (`swim_environment_dataset.csv`), parse times, and save a combined dataset for analysis.

In [9]:
import sys
import sqlite3
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
DB_PATH = RAW / "swim_db.sqlite"
ENV_CSV = PROCESSED / "swim_environment_dataset.csv"
OUT_CSV = PROCESSED / "results_with_environment.csv"

In [10]:
def time_str_to_seconds(s: str) -> float | None:
    """Parse time_str (e.g. '52.00', '1:52.00') to seconds."""
    if pd.isna(s) or not str(s).strip():
        return None
    s = str(s).strip()
    parts = s.replace(",", ".").split(":")
    try:
        if len(parts) == 1:
            return float(parts[0])
        if len(parts) == 2:
            return 60 * float(parts[0]) + float(parts[1])
        if len(parts) == 3:
            return 3600 * float(parts[0]) + 60 * float(parts[1]) + float(parts[2])
    except ValueError:
        pass
    return None

# Load environment (one row per meet)
env = pd.read_csv(ENV_CSV)
print("Environment:", env.shape, "meets")

# Load results from SQLite (all meets)
sql = """
SELECT c.meet_id, r.person_id, r.race_start_time, r.discipline_name, r.gender, r.phase, r.points, r.time_str, r.rank, r.athlete_name, r.country_code
FROM results r
JOIN competitions c ON c.competition_id = r.competition_id
ORDER BY c.meet_id, r.discipline_name, r.phase, r.rank
"""
with sqlite3.connect(DB_PATH) as conn:
    results = pd.read_sql_query(sql, conn)
print("Results:", results.shape, "rows")

Environment: (23, 17) meets
Results: (421531, 11) rows


In [11]:
# Parse time to seconds
results = results[results["phase"] == "Finals"].copy()
results = results[results["points"].notna()].copy()
print(f"Rows with Finals + points: {len(results)}")
results["time_seconds"] = results["time_str"].map(time_str_to_seconds)
valid_time = results["time_seconds"].notna()
print(f"(Optional) Rows with valid time_str: {valid_time.sum()} / {len(results)}")

# Join with environment (inner: only meets that have env data, e.g. excludes Fukuoka)
env_cols = [c for c in env.columns if c != "meet_id"]
merged = results.merge(env[["meet_id"] + env_cols], on="meet_id", how="inner")
print("Merged:", merged.shape)
merged.head(10)

Rows with Finals + points: 38252
(Optional) Rows with valid time_str: 38235 / 38252
Merged: (4846, 28)


,meet_id,person_id,race_start_time,discipline_name,gender,phase,points,time_str,rank,athlete_name,...,elevation,timezone,temperature,dewpoint,wind_speed,pressure,relative_humidity,wbgt,city,country
0,oly_2008,93b926bc-5bb2-41a1-8116-68407f294686,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,1025.0,52.54,1.0,Aaron PEIRSOL,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
1,oly_2008,18fbe8bb-dd04-4f33-a4af-533ff6a02f9c,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,992.0,53.11,2.0,Matt GREVERS,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
2,oly_2008,e5312f2e-d574-440e-9bd1-026e8af0bf90,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,988.0,53.18,3.0,Arkady VYATCHANIN,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
3,oly_2008,a42aa3f5-7f2c-4cba-938b-031b06b5c4d4,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,988.0,53.18,3.0,Hayden STOECKEL,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
4,oly_2008,91dd7ae2-4c45-4975-8f79-dc808ddf3fd1,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,981.0,53.31,5.0,Ashley DELANEY,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
5,oly_2008,b5d6158d-98c7-4b3e-9066-9da2414751d9,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,977.0,53.39,6.0,Liam TANCOCK,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
6,oly_2008,561c3666-e9e3-481a-8e2f-f606fffaa848,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,970.0,53.51,7.0,Aschwin WILDEBOER FABER,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
7,oly_2008,dd12e5d0-1364-452a-917e-e122ed9f02dd,2008-08-12T02:31:00,Men's 100m Backstroke,Men,Finals,944.0,53.99,8.0,Junichi MIYASHITA,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
8,oly_2008,37f64cfc-0948-4610-acf4-f9b900937ae6,2008-08-11T02:30:00,Men's 100m Breaststroke,Men,Finals,1035.0,58.91,1.0,Kosuke KITAJIMA,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China
9,oly_2008,7bf69e5e-5f52-4d6a-a515-e99e212b1caf,2008-08-11T02:30:00,Men's 100m Breaststroke,Men,Finals,1020.0,59.20,2.0,Alexander DALE OEN,...,43,Asia/Shanghai,24.511201,19.36166,1.283588,996.20059,73.07979,25.211736,Beijing,China


In [12]:
PROCESSED.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV}")
merged.groupby("meet_id").size().describe()

Wrote /Users/huangrh/workspace/serena/swim-performance/data/processed/results_with_environment.csv


count     16.000000
mean     302.875000
std      109.534089
min       80.000000
25%      254.750000
50%      317.000000
75%      335.250000
max      558.000000
dtype: float64